In [1]:
import numpy as np
import pandas as pd 
from numpy.polynomial.hermite import hermgauss
from IPython.display import Math
from pathlib import Path

# Step 1: Data Processing 

In [2]:
# Read the .csv file 

from pathlib import Path

DATA_PATH = Path("../data/Bank.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Place Bank.csv inside the data/ folder. "
        "The dataset is not included in this repository."
    )

df = pd.read_csv(DATA_PATH)
df.head(5)

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Step 1.1: Selecting X and y variables from drawn dataset 

In [3]:
# Assign the X and y variables from the table above 

target = "churn"   # y variable, user assigned
drop_cols = ["customer_id"]   # user assigned

cols = [c for c in df.columns if c not in ([target] + drop_cols)]

X = df.drop(columns=[target] + drop_cols).to_numpy()
y = df[target].to_numpy()

In [4]:
# numpy-friendly table of variables (visuals)

def print_table(X, cols, n=5, width=14):
    header = " | ".join(f"{c:>{width}}" for c in cols)
    print(header)
    print("-" * len(header))
    for i in range(min(n, X.shape[0])):
        print(" | ".join(f"{str(X[i,j])[:width]:>{width}}" for j in range(X.shape[1])))

print_table(X, cols, n=10)

  credit_score |        country |         gender |            age |         tenure |        balance | products_number |    credit_card |  active_member | estimated_salary
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
           619 |         France |         Female |             42 |              2 |            0.0 |              1 |              1 |              1 |      101348.88
           608 |          Spain |         Female |             41 |              1 |       83807.86 |              1 |              0 |              1 |      112542.58
           502 |         France |         Female |             42 |              8 |       159660.8 |              3 |              1 |              0 |      113931.57
           699 |         France |         Female |             39 |              1 |            0.0 |              2 |              0 |              0 |  

## Step 1.2: Classify variable X -- Continuous, Categorical and Binary  

In [5]:
# Variable Class Classifier

def variable_class(X, colnames=None, cat_max=20, cat_ratio=0.02, force_cont=()):
    X = np.asarray(X); n, d = X.shape
    bin_idx, cat_idx, cont_idx, report = [], [], [], []

    for j in range(d):
        col = X[:, j]
        u = np.unique(col); k = u.size; r = k / n
        try: col.astype(float); num = True
        except (ValueError, TypeError): num = False

        name = colnames[j] if colnames is not None else f"col_{j}"

        if name in force_cont and num:
            t = "continuous"
        else:
            t = "binary" if k == 2 else "categorical" if (not num) or (k <= cat_max) or (r <= cat_ratio) else "continuous"

        (bin_idx if t=="binary" else cat_idx if t=="categorical" else cont_idx).append(j)
        report.append((j, name, t, k, r, num, u[:5].tolist()))

    return bin_idx, cat_idx, cont_idx, report

# Visuals for Variable Class Classifier

def print_report(report):
    print(f"{'j':>2} {'name':<18} {'type':<12} {'nuniq':>6} {'u/n':>7} {'num':>5}  sample")
    for j, name, t, k, r, num, samp in report:
        print(f"{j:>2} {name:<18} {t:<12} {k:>6} {r:>7.4f} {str(num):>5}  {samp}")

# Call the functions defined previously 

bin_idx, cat_idx, cont_idx, report = variable_class(X, colnames=cols, force_cont=("age","tenure")) # we force "age" and "tenure" as they appear as categorical but act as continuous
print_report(report)

 j name               type          nuniq     u/n   num  sample
 0 credit_score       continuous      460  0.0460  True  [350, 351, 358, 359, 363]
 1 country            categorical       3  0.0003 False  ['France', 'Germany', 'Spain']
 2 gender             binary            2  0.0002 False  ['Female', 'Male']
 3 age                continuous       70  0.0070  True  [18, 19, 20, 21, 22]
 4 tenure             continuous       11  0.0011  True  [0, 1, 2, 3, 4]
 5 balance            continuous     6382  0.6382  True  [0.0, 3768.69, 12459.19, 14262.8, 16893.59]
 6 products_number    categorical       4  0.0004  True  [1, 2, 3, 4]
 7 credit_card        binary            2  0.0002  True  [0, 1]
 8 active_member      binary            2  0.0002  True  [0, 1]
 9 estimated_salary   continuous     9999  0.9999  True  [11.58, 90.07, 91.75, 96.27, 106.67]


## Step 1.3: Assign X_bin, X_cat, X_cont (yet to be standardised/encoded)

In [6]:
# X variables for Binary, Categorical and Continuous 
# and the respective columns (variables and column names have the same order)

X_bin, X_cat, X_cont = X[:, bin_idx], X[:, cat_idx], X[:, cont_idx]
bin_cols, cat_cols, cont_cols = [cols[i] for i in bin_idx], [cols[i] for i in cat_idx], [cols[i] for i in cont_idx]

# Visuals of Binary, Categorical and Continuous

def print_shapes_table(**arrays):
    print(f"{'name':<14} {'shape':<14} {'dtype':<10}")
    print("-" * 40)
    for name, a in arrays.items():
        print(f"{name:<14} {str(a.shape):<14} {str(a.dtype):<10}")

# visuals

print_shapes_table(
    X_bin = X_bin,
    X_cat = X_cat, 
    X_cont = X_cont
)

name           shape          dtype     
----------------------------------------
X_bin          (10000, 3)     object    
X_cat          (10000, 2)     object    
X_cont         (10000, 5)     object    


# Step 2: Train Test Split 

In [7]:
# Train Test Split algorithm

test_size = 0.2
n_samples = len(X)
n_test = int(test_size * n_samples) 

shuffle = np.random.permutation(n_samples)

X_train = X[shuffle[:-n_test]]
y_train = y[shuffle[:-n_test]]
X_test = X[shuffle[-n_test:]]
y_test = y[shuffle[-n_test:]]

# Type specific train test split (binary, categorical, continuous)

def split_by_type(X, bin_idx, cat_idx, cont_idx):
    return X[:, bin_idx], X[:, cat_idx], X[:, cont_idx]

X_train_bin, X_train_cat, X_train_cont = split_by_type(X_train, bin_idx, cat_idx, cont_idx)
X_test_bin,  X_test_cat,  X_test_cont  = split_by_type(X_test,  bin_idx, cat_idx, cont_idx)

# visuals

print_shapes_table(
    X_train_bin=X_train_bin,
    X_train_cat=X_train_cat,
    X_train_cont=X_train_cont,
    y_train=y_train
)

name           shape          dtype     
----------------------------------------
X_train_bin    (8000, 3)      object    
X_train_cat    (8000, 2)      object    
X_train_cont   (8000, 5)      object    
y_train        (8000,)        int64     


### Important Note: At this stage, the Train Values are still "objects" since contain non-integer values like "France" in column "country"

## Step 2.1: Standardise/Encode variables 

In [8]:
# Binary and Categorical variables encoding (convert 'words' into 'numbers')

# Continuous Variable Standardisation 
def standardiser(X_train_cont, X_test_cont, eps = 1e-12):
    Xtr = np.asarray(X_train_cont, float)
    Xte = np.array(X_test_cont, float)
    mu = Xtr.mean(axis = 0)
    var = ((Xtr - mu)**2).mean(axis=0)
    sd = np.sqrt(var)
    Xtr_z = (Xtr - mu) / sd 
    Xte_z = (Xte - mu) / sd
    return Xtr_z, Xte_z, mu, sd

X_train_cont, X_test_cont, mu_cont, sd_cont = standardiser(X_train_cont, X_test_cont)

mf = {"Female":0.0, "Male":1.0}

# overwrite (binary)
X_train_bin = np.vectorize(lambda v: mf.get(v, v))(X_train_bin).astype(float)
X_test_bin  = np.vectorize(lambda v: mf.get(v, v))(X_test_bin ).astype(float)

# overwrite (categorical -> one-hot)
def categorical_encoder(X_train_cat_raw, X_test_cat_raw):
    Xtr = np.asarray(X_train_cat_raw)
    Xte = np.asarray(X_test_cat_raw)

    levels = [np.unique(Xtr[:, j]) for j in range(Xtr.shape[1])]

    Xtr_oh = np.concatenate([(Xtr[:, [j]] == lev).astype(float)
                             for j in range(Xtr.shape[1])
                             for lev in levels[j]], axis=1)

    Xte_oh = np.concatenate([(Xte[:, [j]] == lev).astype(float)
                             for j in range(Xte.shape[1])
                             for lev in levels[j]], axis=1)
    return Xtr_oh, Xte_oh

X_train_cat, X_test_cat = categorical_encoder(X_train_cat, X_test_cat)

In [9]:
def print_shapes_table(**arrays):
    print(f"{'name':<15} {'shape':<12} {'dtype':<10} {'min':>10} {'max':>10}")
    print("-"*62)
    for name, A in arrays.items():
        A = np.asarray(A)
        shp = str(A.shape)
        dt  = str(A.dtype)
        # min/max only for numeric arrays
        if np.issubdtype(A.dtype, np.number) and A.size:
            mn = float(np.min(A))
            mx = float(np.max(A))
            print(f"{name:<15} {shp:<12} {dt:<10} {mn:>10.4f} {mx:>10.4f}")
        else:
            print(f"{name:<15} {shp:<12} {dt:<10} {'-':>10} {'-':>10}")

print_shapes_table(
    X_train_bin=X_train_bin,
    X_train_cat=X_train_cat,
    X_train_cont=X_train_cont,
    y_train=y_train,
    X_test_bin=X_test_bin,
    X_test_cat=X_test_cat,
    X_test_cont=X_test_cont,
    y_test=y_test,
)

name            shape        dtype             min        max
--------------------------------------------------------------
X_train_bin     (8000, 3)    float64        0.0000     1.0000
X_train_cat     (8000, 7)    float64        0.0000     1.0000
X_train_cont    (8000, 5)    float64       -3.1214     5.0722
y_train         (8000,)      int64          0.0000     1.0000
X_test_bin      (2000, 3)    float64        0.0000     1.0000
X_test_cat      (2000, 7)    float64        0.0000     1.0000
X_test_cont     (2000, 5)    float64       -3.1214     5.0722
y_test          (2000,)      int64          0.0000     1.0000


# Step 3: Compute Kernels for Continuous, Categorical and Binary Variables

## Continuous Kernel -- Matern 3/2

In [10]:
display(Math(r"r(x,x') = \sqrt{\sum_{q=1}^d \frac{(x_q - x'_q)^2}{\ell_q^2}}, \;  k(x,x') = \sigma^2(1+\sqrt{3}r)\exp(-\sqrt{3}r)"))

<IPython.core.display.Math object>

In [11]:
def matern32(Z1, Z2=None, ell=1.0, sigma2=1.0):
    Z1 = np.atleast_2d(np.asarray(Z1, float))
    Z2 = Z1 if Z2 is None else np.atleast_2d(np.asarray(Z2, float))
    ell = np.asarray(ell, float); Z1, Z2 = Z1/ell, Z2/ell
    sq = (Z1*Z1).sum(1)[:,None] + (Z2*Z2).sum(1)[None,:] - 2*Z1@Z2.T
    r  = np.sqrt(np.maximum(sq, 0.0)); t = np.sqrt(3.0)*r
    return sigma2*(1.0+t)*np.exp(-t)

Kc = matern32(X_train_cont, ell=np.ones(5), sigma2=1.0) # Matern Kernel -- Kc
Kc += 1e-6*np.eye(Kc.shape[0])   

## Matern 3/2 -- Sanity Checks

In [12]:
K = Kc  # or your full K

print("shape:", K.shape)
print("finite:", np.isfinite(K).all())
print("symmetry max abs:", np.max(np.abs(K - K.T)))
print("diag min/max:", np.min(np.diag(K)), np.max(np.diag(K)))

off = K[~np.eye(K.shape[0], dtype=bool)]
print("offdiag min/mean/median/max:", off.min(), off.mean(), np.median(off), off.max())

try:
    np.linalg.cholesky(K + 1e-6*np.eye(K.shape[0]))
    print("Cholesky OK")
except np.linalg.LinAlgError:
    print("Cholesky FAILED")

shape: (8000, 8000)
finite: True
symmetry max abs: 0.0
diag min/max: 1.0000009999999893 1.000001
offdiag min/mean/median/max: 4.255848861235912e-06 0.06853601043108068 0.033825053089393535 0.998242121340251
Cholesky OK


In [13]:
print(Kc)

[[1.000001   0.08302914 0.04215783 ... 0.01503858 0.01774759 0.02219736]
 [0.08302914 1.000001   0.22710187 ... 0.14707701 0.0278153  0.03162271]
 [0.04215783 0.22710187 1.000001   ... 0.11836298 0.07016554 0.0382121 ]
 ...
 [0.01503858 0.14707701 0.11836298 ... 1.000001   0.08709187 0.15323448]
 [0.01774759 0.0278153  0.07016554 ... 0.08709187 1.000001   0.27862344]
 [0.02219736 0.03162271 0.0382121  ... 0.15323448 0.27862344 1.000001  ]]


## Categorical Variable 

In [14]:
display(Math(r"k_{cat}^{(j)}(u_{i,j},u_{k,j})= \delta(u_{i,j},u_{k,j})= \begin{cases} 1,& u_{i,j}=u_{k,j}\\ 0,& u_{i,j} \neq u_{k,j} \end{cases}"))

display(Math(r"k_{cat}(u_i,u_k)=\sum_{j=1}^J w_j \delta(u_{i,j}, u_{k,j}), \; \; w_j \ge 0"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [15]:
def onehot_7_to_U2(U7):
    U7 = np.asarray(U7)
    A, B = U7[:, :3], U7[:, 3:]  # country(3), products_number(4)

    # strict one-hot validation
    if not (np.all((A == 0) | (A == 1)) and np.all((B == 0) | (B == 1))):
        raise ValueError("One-hot blocks must contain only 0/1.")
    if not (np.all(A.sum(1) == 1) and np.all(B.sum(1) == 1)):
        raise ValueError("Invalid one-hot: each block must have exactly one '1' per row.")

    return np.column_stack([A.argmax(1), B.argmax(1)]).astype(int)

def cat_kernel(U1, U2 = None, w = None):
    U1 = np.asarray(U1)
    if U1.ndim == 1:
        U1 = U1[:, None]
    m, J = U1.shape

    U2 = U1 if U2 is None else np.asarray(U2) 
    if U2.ndim == 1:
        U2 = U2[:, None]
    n, J2 = U2.shape
    if J2 != J:
        raise ValueError(f"U1 and U2 must have same Categorical Columns (got {J} vs {J2})")

    w = np.ones(J, float) if w is None else np.asarray(w, float)
    if w.shape != (J,) or np.any(w < 0):
        raise ValueError(f"w must have shape ({J},) and be nonnegative")

    K = np.zeros((m, n), float)
    for j in range(J):
        K += w[j] * (U1[:, [j]] == U2[:, [j]].T)
    return K 

U_train = onehot_7_to_U2(X_train_cat)
U_test = onehot_7_to_U2(X_test_cat)

w_cat = np.ones(U_train.shape[1])

Kcat = cat_kernel(U_train, w=w_cat)
Kcat_star = cat_kernel(U_test, U_train, w=w_cat)

## Categorical Variable -- Sanity Checks

In [16]:
print("U_train shape:", U_train.shape)
print("U_country levels:", np.unique(U_train[:,0]))
print("U_prod levels:", np.unique(U_train[:,1]))
print("Kcat unique:", np.unique(Kcat))
print("diag unique:", np.unique(np.diag(Kcat)))
print("sym:", np.max(np.abs(Kcat - Kcat.T)))
print(f"Kcat shape: {Kcat.shape}")

try:
    np.linalg.cholesky(Kcat + 1e-6*np.eye(Kcat.shape[0]))
    print("Cholesky OK")
except np.linalg.LinAlgError:
    print("Cholesky FAILED")

U_train shape: (8000, 2)
U_country levels: [0 1 2]
U_prod levels: [0 1 2 3]
Kcat unique: [0. 1. 2.]
diag unique: [2.]
sym: 0.0
Kcat shape: (8000, 8000)
Cholesky OK


In [17]:
print(Kcat)

[[2. 0. 0. ... 1. 0. 1.]
 [0. 2. 2. ... 1. 1. 1.]
 [0. 2. 2. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 2. 1. 0.]
 [0. 1. 1. ... 1. 2. 0.]
 [1. 1. 1. ... 0. 0. 2.]]


## Binary Kernel 

In [18]:
display(Math(r"d_H(b_i,b_k) = \sum_{q=1}^{d_b} \mathbf{1}[b_{i,q} \neq b_{k,q}]"))

display(Math(r"k_b(b_i,b_j) = \sigma_b^2 \exp(-\lambda d_H(b_i,b_k))"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [19]:
def bin_hamming_kernel(B1, B2=None, lam=1.0, sigma2=1.0):
    """
    Exponential Hamming kernel for binary matrices.
    B1: (n1, d) in {0,1}
    B2: (n2, d) in {0,1} or None -> B2=B1
    """
    B1 = np.atleast_2d(np.asarray(B1))
    B2 = B1 if B2 is None else np.atleast_2d(np.asarray(B2))

    # ensure 0/1
    if not (np.all((B1 == 0) | (B1 == 1)) and np.all((B2 == 0) | (B2 == 1))):
        raise ValueError("Binary kernel expects inputs in {0,1}.")

    # Hamming distance matrix: d_H(i,j) = sum_q [B1[i,q] != B2[j,q]]
    dH = (B1[:, None, :] != B2[None, :, :]).sum(axis=2).astype(float)

    return sigma2 * np.exp(-lam * dH)

Kb = bin_hamming_kernel(X_train_bin, lam=1.0, sigma2=1.0)

## Binary Kernel -- Sanity Checks

In [20]:
K = Kb
print("shape:", K.shape)
print("finite:", np.isfinite(K).all())
print("sym:", np.max(np.abs(K - K.T)))
print("diag unique:", np.unique(np.diag(K))[:5])

off = K[~np.eye(K.shape[0], dtype=bool)]
print("offdiag min/mean/median/max:", off.min(), off.mean(), np.median(off), off.max())

try:
    np.linalg.cholesky(K + 1e-6*np.eye(K.shape[0]))
    print("Cholesky OK")
except np.linalg.LinAlgError:
    print("Cholesky FAILED")

shape: (8000, 8000)
finite: True
sym: 0.0
diag unique: [1.]
offdiag min/mean/median/max: 0.049787068367863944 0.34651792594150077 0.36787944117144233 1.0
Cholesky OK


## Create the Master Kernel 

In [21]:
## Check the shapes of all 3 kernels 
print(f"Kernels - Continuous: {Kc.shape}; Categorical: {Kcat.shape}; Binary: {Kb.shape}")
Km = Kc + Kcat + Kb
print(Km)

Kernels - Continuous: (8000, 8000); Categorical: (8000, 8000); Binary: (8000, 8000)
[[4.000001   0.13281621 1.04215783 ... 1.38291802 0.38562703 1.39007681]
 [0.13281621 4.000001   2.27688894 ... 1.28241229 1.16315059 1.16695799]
 [1.04215783 2.27688894 4.000001   ... 1.48624242 1.43804499 1.40609154]
 ...
 [1.38291802 1.28241229 1.48624242 ... 4.000001   2.08709187 1.15323448]
 [0.38562703 1.16315059 1.43804499 ... 2.08709187 4.000001   1.27862344]
 [1.39007681 1.16695799 1.40609154 ... 1.15323448 1.27862344 4.000001  ]]


# Step 4: Gaussian Process Classification 

In [22]:
def sigmoid(x):
    x = np.asarray(x, float)
    out = np.empty_like(x)
    pos = x >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    ex = np.exp(x[~pos])
    out[~pos] = ex / (1.0 + ex)
    return out

def laplace_inference_logistic(K, y, jitter=1e-6, max_iter=50, tol=1e-6):
    K = np.asarray(K, float)
    y = np.asarray(y, float).reshape(-1)
    n = K.shape[0]
    if K.shape != (n, n): raise ValueError("K must be (n,n)")
    if y.shape != (n,):   raise ValueError("y must be (n,) aligned with K")
    if not np.all((y == 0) | (y == 1)): raise ValueError("y must be in {0,1}")

    K_stable = 0.5*(K + K.T) + jitter*np.eye(n)

    f = np.zeros(n)
    for _ in range(max_iter):
        pi = sigmoid(f)
        W  = pi*(1.0-pi)
        Wsqrt = np.sqrt(W)

        # B = I + W^{1/2} K W^{1/2}
        B = np.eye(n) + (Wsqrt[:,None] * (K_stable * Wsqrt[None,:]))
        L_B = np.linalg.cholesky(B)

        b = W*f + (y - pi)

        # a = b - W^{1/2} B^{-1} W^{1/2} K b   (stable form)
        tmp = Wsqrt * (K_stable @ b)
        v = np.linalg.solve(L_B, tmp)
        v = np.linalg.solve(L_B.T, v)
        a = b - Wsqrt * v

        f_new = K_stable @ a
        if np.max(np.abs(f_new - f)) < tol:
            f = f_new
            break
        f = f_new

    pi = sigmoid(f)
    W  = pi*(1.0-pi)
    Wsqrt = np.sqrt(W)
    B = np.eye(n) + (Wsqrt[:,None] * (K_stable * Wsqrt[None,:]))
    L_B = np.linalg.cholesky(B)

    # recompute a one last time at the final f
    b = W*f + (y - pi)
    tmp = Wsqrt * (K_stable @ b)
    v = np.linalg.solve(L_B, tmp)
    v = np.linalg.solve(L_B.T, v)
    a = b - Wsqrt * v

    return f, pi, W, a, L_B, K_stable

## Laplace Inference

In [23]:
f_hat, pi, W, a, L_B, Km_stable = laplace_inference_logistic(Km, y_train, jitter=1e-6, max_iter=50, tol=1e-6)

In [24]:
print(f"f_hat shape: {f_hat.shape}, min/max: {f_hat.min()}, {f_hat.max()}")
print(f"pi shape: {pi.shape}, min/max: {pi.min()}, {pi.max()}")
print(f"W shape: {W.shape}, min/max: {W.min()}, {W.max()}")
print(f"a shape: {a.shape}, min/max: {a.min()}, {a.max()}")
print(f"L_B shape: {L_B.shape}, min/max: {L_B.min()}, {L_B.max()}")
print(f"Km_stable shape: {Km_stable.shape}, min/max: {Km_stable.min()}, {Km_stable.max()}")

f_hat shape: (8000,), min/max: -5.625779424335249, 6.255639107333228
pi shape: (8000,), min/max: 0.003590812785131551, 0.9980840790650953
W shape: (8000,), min/max: 0.001912250181875863, 0.24999999141894189
a shape: (8000,), min/max: -0.9546785016980284, 0.9942998443564754
L_B shape: (8000, 8000), min/max: -0.12022824336404789, 1.3798177916830854
Km_stable shape: (8000, 8000), min/max: 0.049793458257608884, 4.000002


## Computing the Test_Train Kernels

### Continuous -- Matern 3/2

In [25]:
Kc_star = matern32(X_test_cont, X_train_cont, ell=np.ones(5), sigma2=1.0)
print(Kc_star)
print(Kc_star.shape)

[[0.05164653 0.37642569 0.62841301 ... 0.25685013 0.0934096  0.0755093 ]
 [0.02022655 0.09599483 0.1958183  ... 0.23372813 0.0493617  0.08063775]
 [0.09253558 0.19546343 0.06942919 ... 0.07576459 0.01170214 0.03186532]
 ...
 [0.00615036 0.0290496  0.01992056 ... 0.10322537 0.01072026 0.0297389 ]
 [0.05140769 0.02789562 0.01622131 ... 0.00533029 0.00955155 0.00433634]
 [0.0662155  0.09460641 0.04514484 ... 0.09302409 0.02756908 0.06145412]]
(2000, 8000)


### Categorical 

In [26]:
Kcat_star = cat_kernel(U_test, U_train, w=w_cat)
print(Kcat_star)
print(Kcat_star.shape)

[[2. 0. 0. ... 1. 0. 1.]
 [1. 1. 1. ... 2. 1. 0.]
 [1. 1. 1. ... 2. 1. 0.]
 ...
 [1. 0. 0. ... 1. 0. 0.]
 [1. 0. 0. ... 1. 0. 0.]
 [2. 0. 0. ... 1. 0. 1.]]
(2000, 8000)


### Binary 

In [27]:
Kb_star = bin_hamming_kernel(X_test_bin, X_train_bin, lam = 1.0, sigma2 = 1.0)
print(Kb_star)
print(Kb_star.shape)

[[0.36787944 0.13533528 0.36787944 ... 0.13533528 0.13533528 0.13533528]
 [0.13533528 0.36787944 0.13533528 ... 0.36787944 0.36787944 0.36787944]
 [0.36787944 0.13533528 0.36787944 ... 1.         1.         1.        ]
 ...
 [0.13533528 0.36787944 0.13533528 ... 0.36787944 0.36787944 0.36787944]
 [0.36787944 0.13533528 0.36787944 ... 1.         1.         1.        ]
 [0.36787944 0.13533528 0.36787944 ... 0.13533528 0.13533528 0.13533528]]
(2000, 8000)


### Master Kernel for Test-Train Kernels 

In [28]:
Km_star = Kc_star + Kcat_star + Kb_star 
print(Km_star)
print(Km_star.shape)

[[2.41952598 0.51176097 0.99629245 ... 1.39218541 0.22874488 1.21084458]
 [1.15556183 1.46387427 1.33115358 ... 2.60160757 1.41724114 0.4485172 ]
 [1.46041502 1.33079872 1.43730863 ... 3.07576459 2.01170214 1.03186532]
 ...
 [1.14148564 0.39692904 0.15525584 ... 1.47110481 0.3785997  0.39761834]
 [1.41928713 0.1632309  0.38410075 ... 2.00533029 1.00955155 1.00433634]
 [2.43409494 0.22994169 0.41302429 ... 1.22835937 0.16290437 1.1967894 ]]
(2000, 8000)


### k_ss -- Diagonal Kernel 

In [29]:
m = X_test_cont.shape[0]
k_ss = np.zeros(m)

# hardcode the sigma2 values for continuous and binary kernels
sigma2_c = 1.0
sigma2_b = 1.0 

# continuous part: k(x, x) for each test x
k_ss += np.diag(matern32(X_test_cont, X_test_cont, ell=1.0, sigma2=sigma2_c))

# binary part
k_ss += np.diag(bin_hamming_kernel(X_test_bin, X_test_bin, lam=1.0, sigma2 = sigma2_b))

# categorical part 
k_ss += np.diag(cat_kernel(U_test, U_test, w = w_cat))

print(k_ss)
print(k_ss.shape)

[4. 4. 4. ... 4. 4. 4.]
(2000,)


## Laplace Prediciton Logit GPML 

In [30]:
def logistic_gauss_hermite(m, v, n_gh=20):
    m = np.asarray(m, float)
    v = np.asarray(v, float)
    t, w = hermgauss(n_gh)
    z = m[..., None] + np.sqrt(2.0 * v[..., None]) * t[None, :]
    return (w[None, :] * (1.0 / (1.0 + np.exp(-z)))).sum(axis=-1) / np.sqrt(np.pi)

def laplace_predict_logistic_gpml(K_train_stable, K_star, k_ss, a, W_diag, L_B,
                                 jitter=1e-10, n_gh=20, return_debug=False):
    K_train_stable = np.asarray(K_train_stable, float)
    K_star = np.atleast_2d(np.asarray(K_star, float))
    k_ss = np.asarray(k_ss, float).reshape(-1)
    W_diag = np.asarray(W_diag, float).reshape(-1)

    m, n = K_star.shape
    if K_train_stable.shape != (n, n):
        raise ValueError("K_train_stable shape mismatch")
    if k_ss.shape != (m,):
        raise ValueError("k_ss must be (m,)")

    # mean: m_* = K_* a
    m_lat = K_star @ a

    # alpha = K^{-1} K_*^T  (n,m)
    L_K = np.linalg.cholesky(0.5*(K_train_stable + K_train_stable.T) + jitter*np.eye(n))
    alpha = np.linalg.solve(L_K, K_star.T)
    alpha = np.linalg.solve(L_K.T, alpha)

    # term1 = diag(K_* K^{-1} K_*^T)
    term1 = (K_star.T * alpha).sum(axis=0)

    # term2 = diag((W^{1/2}alpha)^T B^{-1} (W^{1/2}alpha))
    Wsqrt = np.sqrt(W_diag)[:, None]
    V = Wsqrt * alpha
    tmp = np.linalg.solve(L_B, V)
    tmp = np.linalg.solve(L_B.T, tmp)
    term2 = (V * tmp).sum(axis=0)

    v_lat = k_ss - term1 + term2
    v_lat = np.maximum(v_lat, 1e-12)

    p1 = logistic_gauss_hermite(m_lat, v_lat, n_gh=n_gh)

    if return_debug:
        return m_lat, v_lat, p1, term1, term2
    return m_lat, v_lat, p1

m_lat, v_lat, p1, term1, term2 = laplace_predict_logistic_gpml(
    Km_stable, Km_star, k_ss, a, W, L_B, return_debug=True
)

print("k_ss min/max:", k_ss.min(), k_ss.max())
print("term1 min/max:", term1.min(), term1.max())
print("term2 min/max:", term2.min(), term2.max())
print("frac term2>term1:", np.mean(term2 > term1 + 1e-8))
print("v_lat min/median/max:", v_lat.min(), np.median(v_lat), v_lat.max())
print("frac v_lat<0:", np.mean(v_lat < -1e-10))
print("min(k_ss - term1):", (k_ss - term1).min())


k_ss min/max: 3.9999999999999893 4.0
term1 min/max: 3.054202390693631 3.9999939683163506
term2 min/max: 0.0056635548812613 0.4645149625664298
frac term2>term1: 0.0
v_lat min/median/max: 0.009749759122058055 0.12654683758937135 0.9536574328828689
frac v_lat<0: 0.0
min(k_ss - term1): 6.031683648544117e-06


### Previously we have drawn p1 values. After that we use them to compute y_pred 

In [31]:
y_prob = p1
y_pred = (y_prob >= 0.5).astype(int)

logloss = -np.mean(y_test * np.log(y_prob + 1e-12) + (1 - y_test)*np.log(1 - y_prob + 1e-12))
acc = np.mean(y_pred == y_test)

# Step 5: Assessment of the Model 

In [32]:
# Statistics
TP = np.sum((y_pred==1) & (y_test==1))
TN = np.sum((y_pred==0) & (y_test==0))
FP = np.sum((y_pred==1) & (y_test==0))
FN = np.sum((y_pred==0) & (y_test==1))
print("TN FP\nFN TP")
print(np.array([[TN, FP],[FN, TP]]))

p_base = np.full_like(y_test, y_train.mean(), dtype=float)
logloss_base = -np.mean(y_test*np.log(p_base+1e-12)+(1-y_test)*np.log(1-p_base+1e-12))
print("logloss model:", logloss, " baseline:", logloss_base)

# AUC = P(score_pos > score_neg) + 0.5*P(tie)
pos = y_prob[y_test==1]
neg = y_prob[y_test==0]
auc = (pos[:,None] > neg[None,:]).mean() + 0.5*(pos[:,None] == neg[None,:]).mean()
print("AUC:", auc)

ths = np.linspace(0.05, 0.95, 19)
best = None
for t in ths:
    yp = (y_prob >= t).astype(int)
    acc_t = np.mean(yp == y_test)
    if best is None or acc_t > best[0]:
        best = (acc_t, t)
print("best acc:", best[0], "at threshold:", best[1])

print("m_lat min/med/max:", m_lat.min(), np.median(m_lat), m_lat.max())
print("v_lat min/med/max:", v_lat.min(), np.median(v_lat), v_lat.max())
print("fraction near 0.5:", np.mean((y_prob > 0.45) & (y_prob < 0.55)))

TN FP
FN TP
[[1556   51]
 [ 221  172]]
logloss model: 0.34353097818377626  baseline: 0.4957627490793536
AUC: 0.8520357025798392
best acc: 0.864 at threshold: 0.49999999999999994
m_lat min/med/max: -5.301207012960105 -2.161120652052152 5.587340398853016
v_lat min/med/max: 0.009749759122058055 0.12654683758937135 0.9536574328828689
fraction near 0.5: 0.034


In [33]:
def quantile_report(probs, name=""):
    q = np.quantile(probs, [0.05, 0.25, 0.5, 0.75, 0.95])
    print(f"{name} q05/q25/q50/q75/q95:", np.round(q, 4))

p0 = y_prob[y_test==0]
p1_ = y_prob[y_test==1]

quantile_report(p0, "true 0 probs")
quantile_report(p1_, "true 1 probs")

# crude overlap indicator: fraction of class-1 probs below median of class-0, and vice versa
med0 = np.median(p0)
med1 = np.median(p1_)
print("frac(class1 <= median(class0)):", np.mean(p1_ <= med0))
print("frac(class0 >= median(class1)):", np.mean(p0 >= med1))


true 0 probs q05/q25/q50/q75/q95: [0.0137 0.0407 0.0843 0.1755 0.4193]
true 1 probs q05/q25/q50/q75/q95: [0.0537 0.2141 0.4363 0.694  0.9238]
frac(class1 <= median(class0)): 0.08651399491094147
frac(class0 >= median(class1)): 0.044803982576229


In [34]:
edges = np.linspace(0, 1, 11)
c0, _ = np.histogram(y_prob[y_test==0], bins=edges)
c1, _ = np.histogram(y_prob[y_test==1], bins=edges)

print("bin      | true0 | true1")
for i in range(10):
    print(f"{edges[i]:.1f}-{edges[i+1]:.1f} | {c0[i]:5d} | {c1[i]:5d}")


bin      | true0 | true1
0.0-0.1 |   906 |    40
0.1-0.2 |   369 |    50
0.2-0.3 |   148 |    55
0.3-0.4 |    90 |    38
0.4-0.5 |    43 |    38
0.5-0.6 |    21 |    39
0.6-0.7 |    18 |    39
0.7-0.8 |     5 |    38
0.8-0.9 |     6 |    28
0.9-1.0 |     1 |    28


In [35]:
def kernel_stats(name, K):
    d = np.diag(K)
    print(f"{name}: diag mean = {d.mean():.3f}, diag min/max = ({d.min():.3f}, {d.max():.3f}), fro = {np.linalg.norm(K):.2e}")

kernel_stats("Kc", Kc)
kernel_stats("Kb", Kb)
kernel_stats("Kcat", Kcat)

Kc: diag mean = 1.000, diag min/max = (1.000, 1.000), fro = 9.42e+02
Kb: diag mean = 1.000, diag min/max = (1.000, 1.000), fro = 3.65e+03
Kcat: diag mean = 2.000, diag min/max = (2.000, 2.000), fro = 8.76e+03


In [37]:
# ============================================================
# Gaussian Process Classifier: Threshold + Probability Diagnostics
# ============================================================
# Purpose:
# - test whether threshold 0.5 is conservative
# - test precision/recall trade-off across thresholds
# - test probability separation between churn and non-churn
# - test ranking quality via AUC
# - test probability quality via log loss
#
# Expected variables from the notebook:
#   y_test : true binary test labels
#   y_prob : predicted probabilities
# If y_prob is not defined, this cell falls back to p1.

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Resolve notebook variables safely
# ------------------------------------------------------------

if "y_prob" in globals():
    prob = np.asarray(y_prob).reshape(-1)
elif "p1" in globals():
    prob = np.asarray(p1).reshape(-1)
else:
    raise NameError("Could not find y_prob or p1. Run the GP prediction cell first.")

if "y_test" not in globals():
    raise NameError("Could not find y_test. Run the train/test split and prediction cells first.")

y_true = np.asarray(y_test).astype(int).reshape(-1)

if len(y_true) != len(prob):
    raise ValueError(
        f"Length mismatch: len(y_test)={len(y_true)}, len(probabilities)={len(prob)}"
    )

if not np.all(np.isfinite(prob)):
    raise ValueError("Predicted probabilities contain NaN or infinite values.")

if prob.min() < 0 or prob.max() > 1:
    print("Warning: probabilities are outside [0, 1]. Clipping for diagnostics.")
    prob = np.clip(prob, 1e-12, 1 - 1e-12)

print("Diagnostic input check")
print("----------------------")
print("n_test:", len(y_true))
print("actual positive rate:", round(float(y_true.mean()), 4))
print("probability min/median/max:", round(float(prob.min()), 4), round(float(np.median(prob)), 4), round(float(prob.max()), 4))


# ------------------------------------------------------------
# 2. Manual metric functions
# ------------------------------------------------------------

def binary_metrics_at_threshold(y_true, prob, threshold=0.5):
    y_pred_t = (prob >= threshold).astype(int)

    tp = int(np.sum((y_true == 1) & (y_pred_t == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred_t == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred_t == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred_t == 0)))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    balanced_accuracy = 0.5 * (recall + specificity)

    return {
        "threshold": float(threshold),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "predicted_positive_rate": float(np.mean(y_pred_t)),
        "actual_positive_rate": float(np.mean(y_true)),
    }


def binary_log_loss(y_true, prob, eps=1e-12):
    p = np.clip(prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))


def rank_auc(y_true, prob):
    """
    Manual ROC-AUC using ranks.
    Equivalent interpretation:
    probability that a randomly chosen positive has a higher score
    than a randomly chosen negative.
    """
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob)

    pos = prob[y_true == 1]
    neg = prob[y_true == 0]

    if len(pos) == 0 or len(neg) == 0:
        return np.nan

    # Pairwise implementation is simple and transparent.
    # Fine for small/medium test sets.
    greater = 0
    equal = 0

    for p in pos:
        greater += np.sum(p > neg)
        equal += np.sum(p == neg)

    return (greater + 0.5 * equal) / (len(pos) * len(neg))


# ------------------------------------------------------------
# 3. Main threshold-0.5 diagnostic
# ------------------------------------------------------------

metrics_05 = binary_metrics_at_threshold(y_true, prob, threshold=0.5)
metrics_05_df = pd.DataFrame([metrics_05]).round(4)

print("\nThreshold = 0.5 metrics")
print("-----------------------")
display(metrics_05_df)


# ------------------------------------------------------------
# 4. Threshold sweep
# ------------------------------------------------------------

thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)

threshold_results = pd.DataFrame(
    [binary_metrics_at_threshold(y_true, prob, threshold=t) for t in thresholds]
)

threshold_view = threshold_results[
    [
        "threshold",
        "precision",
        "recall",
        "f1",
        "balanced_accuracy",
        "predicted_positive_rate",
        "TP",
        "FP",
        "FN",
        "TN",
    ]
].round(4)

print("\nThreshold sweep")
print("---------------")
display(threshold_view)


# ------------------------------------------------------------
# 5. Best thresholds by F1 and balanced accuracy
# ------------------------------------------------------------

best_f1 = threshold_results.loc[threshold_results["f1"].idxmax()]
best_bacc = threshold_results.loc[threshold_results["balanced_accuracy"].idxmax()]

best_thresholds = pd.DataFrame(
    {
        "best_f1_threshold": best_f1,
        "best_balanced_accuracy_threshold": best_bacc,
    }
).round(4)

print("\nBest thresholds")
print("---------------")
display(best_thresholds)


# ------------------------------------------------------------
# 6. Ranking and probability-quality diagnostics
# ------------------------------------------------------------

auc_manual = rank_auc(y_true, prob)
model_logloss = binary_log_loss(y_true, prob)

baseline_prob = np.full_like(prob, fill_value=y_true.mean(), dtype=float)
baseline_logloss = binary_log_loss(y_true, baseline_prob)

quality_summary = pd.DataFrame(
    [
        {
            "AUC": auc_manual,
            "model_log_loss": model_logloss,
            "baseline_log_loss": baseline_logloss,
            "log_loss_improvement": baseline_logloss - model_logloss,
        }
    ]
).round(4)

print("\nRanking and probability-quality summary")
print("---------------------------------------")
display(quality_summary)


# ------------------------------------------------------------
# 7. Probability separation by true class
# ------------------------------------------------------------

prob_non_churn = prob[y_true == 0]
prob_churn = prob[y_true == 1]

quantile_summary = pd.DataFrame(
    {
        "true_0_non_churn": np.quantile(prob_non_churn, [0.05, 0.25, 0.50, 0.75, 0.95]),
        "true_1_churn": np.quantile(prob_churn, [0.05, 0.25, 0.50, 0.75, 0.95]),
    },
    index=["q05", "q25", "q50", "q75", "q95"],
).round(4)

print("\nPredicted-probability quantiles by true class")
print("---------------------------------------------")
display(quantile_summary)


# ------------------------------------------------------------
# 8. Probability-bin diagnostic
# ------------------------------------------------------------

bins = np.linspace(0, 1, 11)
bin_labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins) - 1)]

prob_bins = pd.cut(prob, bins=bins, labels=bin_labels, include_lowest=True)

bin_diagnostic = (
    pd.DataFrame({"y_true": y_true, "prob_bin": prob_bins})
    .groupby("prob_bin", observed=False)
    .agg(
        count=("y_true", "size"),
        churn_count=("y_true", "sum"),
        empirical_churn_rate=("y_true", "mean"),
    )
)

bin_diagnostic["share_of_test_set"] = bin_diagnostic["count"] / len(y_true)
bin_diagnostic = bin_diagnostic.round(4)

print("\nProbability-bin diagnostic")
print("--------------------------")
display(bin_diagnostic)


# ------------------------------------------------------------
# 9. Automated interpretation
# ------------------------------------------------------------

precision_05 = metrics_05["precision"]
recall_05 = metrics_05["recall"]
pred_pos_rate_05 = metrics_05["predicted_positive_rate"]
actual_pos_rate = metrics_05["actual_positive_rate"]

print("\nAutomated interpretation")
print("------------------------")

print(f"At threshold 0.5, precision = {precision_05:.3f} and recall = {recall_05:.3f}.")
print(f"The model predicts positives for {pred_pos_rate_05:.3f} of the test set.")
print(f"The true positive rate in the test set is {actual_pos_rate:.3f}.")

if precision_05 > recall_05:
    print(
        "\nInterpretation: the classifier is precision-oriented at threshold 0.5. "
        "When it predicts churn, it is relatively selective, but it misses a material "
        "number of actual churners."
    )

if pred_pos_rate_05 < actual_pos_rate:
    print(
        "\nThe predicted positive rate is below the actual positive rate, which supports "
        "the claim that threshold 0.5 is conservative."
    )

if best_f1["threshold"] < 0.5:
    print(
        f"\nThe best F1 threshold is {best_f1['threshold']:.2f}, below 0.5. "
        "This means the model's probabilities are useful, but the default threshold is "
        "not necessarily the best decision rule."
    )

if auc_manual >= 0.8:
    print(
        f"\nAUC = {auc_manual:.3f}, indicating strong ranking ability: the model usually "
        "assigns higher churn risk to churners than to non-churners."
    )

if model_logloss < baseline_logloss:
    print(
        f"\nModel log loss = {model_logloss:.3f}, below baseline log loss = {baseline_logloss:.3f}. "
        "This means the predicted probabilities carry useful information beyond the base rate."
    )

Diagnostic input check
----------------------
n_test: 2000
actual positive rate: 0.1965
probability min/median/max: 0.0051 0.1089 0.9959

Threshold = 0.5 metrics
-----------------------


,threshold,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,specificity,f1,predicted_positive_rate,actual_positive_rate
0,0.5,1556,51,221,172,0.864,0.703,0.7713,0.4377,0.9683,0.5584,0.1115,0.1965



Threshold sweep
---------------


,threshold,precision,recall,f1,balanced_accuracy,predicted_positive_rate,TP,FP,FN,TN
0,0.10,0.3349,0.8982,0.4879,0.7310,0.5270,353,701,40,906
1,0.15,0.4077,0.8321,0.5473,0.7682,0.4010,327,475,66,1132
2,0.20,0.4772,0.7710,0.5895,0.7822,0.3175,303,332,90,1275
3,0.25,0.5252,0.6896,0.5963,0.7686,0.2580,271,245,122,1362
4,0.30,0.5741,0.6310,0.6012,0.7583,0.2160,248,184,145,1423
5,0.35,0.6571,0.5852,0.6191,0.7553,0.1750,230,120,163,1487
6,0.40,0.6908,0.5344,0.6026,0.7379,0.1520,210,94,183,1513
7,0.45,0.7315,0.4784,0.5785,0.7177,0.1285,188,69,205,1538
8,0.50,0.7713,0.4377,0.5584,0.7030,0.1115,172,51,221,1556
9,0.55,0.7989,0.3842,0.5189,0.6803,0.0945,151,38,242,1569



Best thresholds
---------------


...


Predicted-probability quantiles by true class
---------------------------------------------


,true_0_non_churn,true_1_churn
q05,0.0137,0.0537
q25,0.0407,0.2141
q50,0.0843,0.4363
q75,0.1755,0.6940
q95,0.4193,0.9238



Probability-bin diagnostic
--------------------------


,count,churn_count,empirical_churn_rate,share_of_test_set
prob_bin,,,,
0.0-0.1,946,40,0.0423,0.4730
0.1-0.2,419,50,0.1193,0.2095
0.2-0.3,203,55,0.2709,0.1015
0.3-0.4,128,38,0.2969,0.0640
0.4-0.5,81,38,0.4691,0.0405
0.5-0.6,60,39,0.6500,0.0300
0.6-0.7,57,39,0.6842,0.0285
0.7-0.8,43,38,0.8837,0.0215
0.8-0.9,34,28,0.8235,0.0170



Automated interpretation
------------------------
At threshold 0.5, precision = 0.771 and recall = 0.438.
The model predicts positives for 0.112 of the test set.
The true positive rate in the test set is 0.197.

Interpretation: the classifier is precision-oriented at threshold 0.5. When it predicts churn, it is relatively selective, but it misses a material number of actual churners.

The predicted positive rate is below the actual positive rate, which supports the claim that threshold 0.5 is conservative.

The best F1 threshold is 0.35, below 0.5. This means the model's probabilities are useful, but the default threshold is not necessarily the best decision rule.

AUC = 0.852, indicating strong ranking ability: the model usually assigns higher churn risk to churners than to non-churners.

Model log loss = 0.344, below baseline log loss = 0.496. This means the predicted probabilities carry useful information beyond the base rate.
